This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of Quickstart on link https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

# Setup Python Paths for Libraries

In [1]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../QFast-SCNN_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/tmp/home_dir/.local/lib/python3.10/site-packages', '/home/jose-vitor/finn-repo/deps/qonnx/src', '/home/jose-vitor/finn-repo/deps/finn-experimental/src', '/home/jose-vitor/finn-repo/deps/brevitas/src', '/home/jose-vitor/finn-repo/deps/pyverilator', '/home/jose-vitor/finn-repo/src', '/usr/local/lib/python3.10/dist-packages', '/workspace/src/dataset-loading', '/usr/lib/python3/dist-packages', '/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment']


# Compare the Outputs of Pytorch Model and QONNX Model

In [7]:
import torch
import models.QFastSCNN as qfscnn
from my_finn_utils import load_state_dict
from config import CROP_SIZE, NUM_CLASSES

pytorch_model = qfscnn.QFastSCNN(NUM_CLASSES)
pytorch_model = load_state_dict(pytorch_model, path="../train_environment/model_weights/quant_params/best_quant_model.pth", strict=True)
pytorch_model.eval()

dummy_input = torch.randn(1, 3, *CROP_SIZE)
dummy_input.size()

Carregando modelo best_quant_model


torch.Size([1, 3, 768, 768])

In [10]:
# Run a foward pass on Pytorch model
with torch.inference_mode():
    pytorch_output = pytorch_model(dummy_input)
pytorch_output.size()

/usr/local/lib/python3.10/dist-packages/torch/overrides.py:1528: DeprecationWarning: Defining your `__torch_function__ as a plain method is deprecated and will be an error in future, please define it as a classmethod.
  warnings.warn("Defining your `__torch_function__ as a plain method is deprecated and "
/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment/models/QFastSCNN.py:195: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.cat([x, feat1, feat2, feat3, feat4], dim=1)


torch.Size([1, 19, 96, 96])

Run a cleanup transformation on the exported QONNX model

In [21]:
import onnx
from finn.util.visualization import showInNetron
from qonnx.util.cleanup import cleanup
from pathlib import Path

FINN_BIT_WIDTH = 8

onnx_path = f'../onnx/quant_model_{FINN_BIT_WIDTH}_bits.onnx'
#onnx_path = os.path.abspath(f'../onnx_models/quant_model_{FINN_BIT_WIDTH}_bits.onnx')
onnx_name = Path(onnx_path).stem
export_onnx_path_cleaned = f'./cleaned_qonnx_models/{onnx_name}-clean.onnx'

cleanup(onnx_path, out_file=export_onnx_path_cleaned)

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


In [22]:
showInNetron(export_onnx_path_cleaned)

Serving '/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/finn_environment/cleaned_qonnx_models/quant_model_8_bits-clean.onnx' at http://0.0.0.0:8081


In [33]:
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
#import onnx.numpy_helper as nph

model = ModelWrapper(export_onnx_path_cleaned)
input_dict = {"global_in": dummy_input.numpy()}
output_dict = oxe.execute_onnx(model, input_dict)
produced_qonnx = output_dict[list(output_dict.keys())[0]]

produced_qonnx

Exception: Found unspecified tensor shapes, try infer_shapes

In [32]:
dummy_input.numpy()

array([[[[-0.0139523 ,  1.0438383 , -0.4399118 , ..., -0.02287943,
          -0.5399219 , -0.24720904],
         [-0.7763864 , -0.686158  ,  0.24802715, ...,  1.8952016 ,
           2.74346   ,  0.5640344 ],
         [-2.552713  , -1.9085989 ,  0.43867692, ..., -1.5664718 ,
          -1.4403528 ,  1.3421234 ],
         ...,
         [ 0.26740304,  0.33051908,  0.08817148, ..., -0.08928256,
          -0.7420384 , -0.5322253 ],
         [-1.0088631 ,  0.06234841,  0.20615102, ..., -0.8540885 ,
          -0.43820652,  0.7244086 ],
         [ 0.67255306,  0.52427036,  0.2744373 , ...,  0.97699416,
          -1.611674  , -1.3763834 ]],

        [[-0.66585386,  1.5861571 ,  0.19723634, ...,  0.4360117 ,
          -0.11678059,  0.34125137],
         [ 1.3520724 ,  2.0562851 , -0.06961559, ..., -1.0995084 ,
           0.7481619 , -0.2515956 ],
         [ 0.48976508, -0.6061622 ,  0.15442045, ...,  0.3803053 ,
           0.2866773 ,  1.4428307 ],
         ...,
         [ 0.08090279,  0.6938996 